# Singular Method Context Pruning Report

This notebook reports singular pruning results by exact context: dataset × model × scope × prune ratio × method.

It uses saved singular pruned checkpoints as the source of truth for structural FLOPs and parameter reduction. Accuracy delta and pruning time are read from the benchmark artifact index. Missing checkpoint contexts are listed explicitly so they can be rerun without contaminating the report.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path.cwd()
REPORT_DIR = PROJECT_ROOT / "report_artifacts" / "singular_method_context_report"
TABLE_DIR = REPORT_DIR / "tables"
PLOT_DIR = REPORT_DIR / "plots"

RUN_SINGULAR_CONTEXT_REPORT_BUILDER = bool(globals().get("RUN_SINGULAR_CONTEXT_REPORT_BUILDER", True))
SHOW_ROWS = int(globals().get("SHOW_ROWS", 80))
MAX_INLINE_CONTEXTS = globals().get("MAX_INLINE_CONTEXTS", "all")

if RUN_SINGULAR_CONTEXT_REPORT_BUILDER:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "build_singular_method_context_report.py"),
        "--report-dir",
        str(REPORT_DIR),
    ]
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode:
        print(result.stderr)
        raise RuntimeError(f"singular context report builder failed with exit code {result.returncode}")

display(Markdown(f"Report directory: `{REPORT_DIR}`"))

In [ ]:
def read_table(name):
    path = TABLE_DIR / name
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()

def show_table(title, df, sort_cols=None, n=SHOW_ROWS):
    display(Markdown(f"### {title}"))
    if df.empty:
        display(Markdown("No rows available."))
        return
    out = df.copy()
    cols = [c for c in (sort_cols or []) if c in out.columns]
    if cols:
        out = out.sort_values(cols, kind="mergesort")
    display(out.head(n))

CONTEXT_SORT = ["dataset", "model", "scope", "ratio", "method"]
index_df = read_table("singular_method_checkpoint_index.csv")
metrics_df = read_table("singular_method_context_metrics.csv")
coverage_df = read_table("singular_method_context_coverage.csv")
missing_df = read_table("singular_method_missing_or_failed_checkpoints.csv")
plot_manifest = read_table("plot_manifest_singular_method_context_metrics.csv")

show_table("Singular checkpoint index", index_df, CONTEXT_SORT)
show_table("Singular method metrics by context", metrics_df, CONTEXT_SORT)
show_table("Context coverage", coverage_df, ["dataset", "model", "scope", "ratio"])
show_table("Missing or failed singular checkpoints", missing_df, CONTEXT_SORT, n=200)

In [ ]:
display(Markdown("## Singular method metric plots by exact context"))
if plot_manifest.empty:
    display(Markdown("No context plots were generated."))
else:
    plots = plot_manifest.sort_values([c for c in ["dataset", "model", "scope", "ratio"] if c in plot_manifest.columns], kind="mergesort").copy()
    if MAX_INLINE_CONTEXTS not in (None, "all", "ALL"):
        plots = plots.head(int(MAX_INLINE_CONTEXTS))
    for _, row in plots.iterrows():
        title = f"{row.get('dataset')} | {row.get('model')} | {row.get('scope')} | r={row.get('ratio')}"
        display(Markdown(f"### {title}"))
        combined = Path(str(row.get("combined_plot", "")))
        if combined.exists():
            display(Image(filename=str(combined)))
        else:
            display(Markdown(f"Missing combined plot: `{combined}`"))
        if not metrics_df.empty:
            mask = (
                metrics_df.get("dataset", pd.Series(dtype=str)).astype(str).eq(str(row.get("dataset")))
                & metrics_df.get("model", pd.Series(dtype=str)).astype(str).eq(str(row.get("model")))
                & metrics_df.get("scope", pd.Series(dtype=str)).astype(str).eq(str(row.get("scope")))
                & pd.to_numeric(metrics_df.get("ratio", pd.Series(dtype=float)), errors="coerce").sub(float(row.get("ratio"))).abs().lt(1e-9)
            )
            cols = [c for c in ["method_display", "method", "accuracy_delta_pp", "direct_flops_reduction_pct", "direct_params_reduction_pct", "time_sec", "metric_status", "checkpoint_path_resolved"] if c in metrics_df.columns]
            display(metrics_df[mask][cols].sort_values([c for c in ["method_display", "method"] if c in cols]).head(80))